# YOLO26 Vessel Detection Experiments with SAHI

**Variables tested:**
- Image size: 512, 640, 768, 1024
- Epochs: 50, 100, 150, 200
- Model size: nano, small, medium, large
- SAHI inference: Compare standard vs SAHI inference for each trained model

## 1. Setup and Configuration

In [1]:
# Install packages if needed (run once, then restart kernel)
import sys
!{sys.executable} -m pip install --user numpy==1.26.4 pandas ultralytics sahi matplotlib --quiet

In [2]:
from ultralytics import YOLO
import torch
import time
import gc
import json
from datetime import datetime
from pathlib import Path
import os

# Verify GPU is available
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA B200
GPU Memory: 191.5 GB


In [3]:
# Check SAHI installation
try:
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction, get_prediction
    SAHI_AVAILABLE = True
    print("SAHI installed and ready")
except ImportError:
    SAHI_AVAILABLE = False
    print("SAHI not installed. Run: pip install sahi")
    print(" SAHI evaluation will be skipped.")

SAHI installed and ready


In [4]:
data_yaml_path = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml"

# For SAHI evaluation - path to validation images
val_images_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/images/val"

# Where to save progress (so we can resume if kernel crashes)
progress_file = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/experiment_progress.json"

# Verify paths exist
for path, name in [(data_yaml_path, "Data config"), (val_images_dir, "Val images")]:
    if Path(path).exists():
        print(f"{name} found: {path}")
    else:
        print(f"{name} NOT found: {path}")

Data config found: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml
Val images found: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/images/val


## 2. Helper Functions

In [5]:
def clear_memory():
    """Clear GPU and CPU memory between experiments."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("   Memory cleared.")


def save_progress(results, filepath):
    """Save experiment progress to JSON file."""
    # Convert any non-serializable objects
    clean_results = []
    for r in results:
        clean_r = {}
        for k, v in r.items():
            if isinstance(v, (int, float, str, bool, type(None))):
                clean_r[k] = v
            else:
                clean_r[k] = str(v)
        clean_results.append(clean_r)
    
    with open(filepath, 'w') as f:
        json.dump(clean_results, f, indent=2)
    print(f"   Progress saved to {filepath}")


def load_progress(filepath):
    """Load previous experiment progress."""
    if Path(filepath).exists():
        with open(filepath, 'r') as f:
            return json.load(f)
    return []


def get_gpu_memory_usage():
    """Get current GPU memory usage."""
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        return f"{used:.1f}/{total:.1f} GB"
    return "N/A"

In [6]:
def evaluate_with_sahi(model_path, val_images_dir, slice_size, overlap_ratio=0.2, conf_thresh=0.25):
    """
    Evaluate a trained model using SAHI sliced inference.
    """
    if not SAHI_AVAILABLE:
        return None
    
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    
    try:
        # Load model for SAHI
        detection_model = AutoDetectionModel.from_pretrained(
            model_type='ultralytics',
            model_path=str(model_path),
            confidence_threshold=conf_thresh,
            device='cuda:0' if torch.cuda.is_available() else 'cpu'
        )
        
        # Get all validation images
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
        image_files = [f for f in Path(val_images_dir).iterdir() 
                       if f.suffix.lower() in image_extensions]
        
        total_detections = 0
        start_time = time.time()
        
        for img_path in image_files:
            result = get_sliced_prediction(
                str(img_path),
                detection_model,
                slice_height=slice_size,
                slice_width=slice_size,
                overlap_height_ratio=overlap_ratio,
                overlap_width_ratio=overlap_ratio,
            )
            total_detections += len(result.object_prediction_list)
        
        elapsed_time = time.time() - start_time
        
        # Clean up
        del detection_model
        clear_memory()
        
        return {
            'total_detections': total_detections,
            'num_images': len(image_files),
            'avg_detections_per_image': total_detections / len(image_files) if image_files else 0,
            'inference_time': elapsed_time,
            'time_per_image': elapsed_time / len(image_files) if image_files else 0
        }
    except Exception as e:
        print(f"   SAHI evaluation failed: {str(e)}")
        return None


def evaluate_standard(model_path, val_images_dir, imgsz, conf_thresh=0.25):
    """
    Evaluate a trained model using standard (non-SAHI) inference.
    """
    try:
        model = YOLO(str(model_path))
        
        # Get all validation images
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
        image_files = [f for f in Path(val_images_dir).iterdir() 
                       if f.suffix.lower() in image_extensions]
        
        total_detections = 0
        start_time = time.time()
        
        for img_path in image_files:
            results = model.predict(
                str(img_path),
                imgsz=imgsz,
                conf=conf_thresh,
                verbose=False
            )
            total_detections += len(results[0].boxes)
        
        elapsed_time = time.time() - start_time
        
        # Clean up
        del model
        clear_memory()
        
        return {
            'total_detections': total_detections,
            'num_images': len(image_files),
            'avg_detections_per_image': total_detections / len(image_files) if image_files else 0,
            'inference_time': elapsed_time,
            'time_per_image': elapsed_time / len(image_files) if image_files else 0
        }
    except Exception as e:
        print(f"   Standard evaluation failed: {str(e)}")
        return None

## 3. Define Base Configuration

In [7]:
# Base configuration
BASE_CONFIG = {
    'data': data_yaml_path,
    'epochs': 100,
    'imgsz': 640,
    'batch': 16,
    'rect': True,
    'device': 0,  # Use GPU
    
    # Augmentations
    'mosaic': 0.5,
    'scale': 0.2,
    'fliplr': 0.5,
    'flipud': 0.0,
    
    # Training settings
    'patience': 50,
    'save': True,
    'plots': True,
    'project': 'jupiter_inlet_experiments_v2',
    
    # Memory optimization
    'workers': 4,  
}

BASE_MODEL = 'yolo26s.pt'

# SAHI settings
SAHI_OVERLAP_RATIO = 0.2
SAHI_CONFIDENCE_THRESHOLD = 0.25

print("Base configuration:")
for k, v in BASE_CONFIG.items():
    if k != 'data':  
        print(f"  {k}: {v}")
print(f"  model: {BASE_MODEL}")

Base configuration:
  epochs: 100
  imgsz: 640
  batch: 16
  rect: True
  device: 0
  mosaic: 0.5
  scale: 0.2
  fliplr: 0.5
  flipud: 0.0
  patience: 50
  save: True
  plots: True
  project: jupiter_inlet_experiments_v2
  workers: 4
  model: yolo26s.pt


## 4. Define Experiments

Each experiment changes **only one variable** from the baseline.

**Note:** Batch sizes are reduced for large images and models to prevent memory crashes.

In [8]:
# Define all experiments
experiments = [
    # ==========================================================================
    # BASELINE
    # ==========================================================================
    {'name': '796img_baseline_s_640_e100', 'model': 'yolo26s.pt', 'changes': {}},
    
    # ==========================================================================
    # IMAGE SIZE EXPERIMENTS (keeping epochs=100, model=small)
    # ==========================================================================
    {'name': '796img_imgsz_512', 'model': 'yolo26s.pt', 'changes': {'imgsz': 512, 'batch': 16}},
    {'name': '796img_imgsz_768', 'model': 'yolo26s.pt', 'changes': {'imgsz': 768, 'batch': 8}},
    {'name': '796img_imgsz_1024', 'model': 'yolo26s.pt', 'changes': {'imgsz': 1024, 'batch': 4}},
    
    # ==========================================================================
    # EPOCH EXPERIMENTS (keeping imgsz=640, model=small)
    # ==========================================================================
    {'name': '796img_epochs_50', 'model': 'yolo26s.pt', 'changes': {'epochs': 50}},
    {'name': '796img_epochs_150', 'model': 'yolo26s.pt', 'changes': {'epochs': 150}},
    {'name': '796img_epochs_200', 'model': 'yolo26s.pt', 'changes': {'epochs': 200}},
    
    # ==========================================================================
    # MODEL SIZE EXPERIMENTS (keeping imgsz=640, epochs=100)
    # ==========================================================================
    {'name': '796img_model_nano', 'model': 'yolo26n.pt', 'changes': {'batch': 16}},
    {'name': '796img_model_medium', 'model': 'yolo26m.pt', 'changes': {'batch': 8}},
    {'name': '796img_model_large', 'model': 'yolo26l.pt', 'changes': {'batch': 4}},
]

print(f"Total experiments defined: {len(experiments)}")
print("\nExperiment list:")
for i, exp in enumerate(experiments, 1):
    changes_str = str(exp['changes']) if exp['changes'] else 'baseline'
    print(f"  {i}. {exp['name']}: {exp['model']} | {changes_str}")

Total experiments defined: 10

Experiment list:
  1. 796img_baseline_s_640_e100: yolo26s.pt | baseline
  2. 796img_imgsz_512: yolo26s.pt | {'imgsz': 512, 'batch': 16}
  3. 796img_imgsz_768: yolo26s.pt | {'imgsz': 768, 'batch': 8}
  4. 796img_imgsz_1024: yolo26s.pt | {'imgsz': 1024, 'batch': 4}
  5. 796img_epochs_50: yolo26s.pt | {'epochs': 50}
  6. 796img_epochs_150: yolo26s.pt | {'epochs': 150}
  7. 796img_epochs_200: yolo26s.pt | {'epochs': 200}
  8. 796img_model_nano: yolo26n.pt | {'batch': 16}
  9. 796img_model_medium: yolo26m.pt | {'batch': 8}
  10. 796img_model_large: yolo26l.pt | {'batch': 4}


## 5. Check for Previous Progress (Resume Support)

In [9]:
# Check for previous progress
previous_results = load_progress(progress_file)

if previous_results:
    completed_experiments = [r['experiment'] for r in previous_results if r.get('status') == 'success']
    print(f"Found previous progress: {len(completed_experiments)} completed experiments")
    for exp_name in completed_experiments:
        print(f"{exp_name}")
    
    # Ask whether to resume or start fresh
    RESUME_FROM_PREVIOUS = True  # Set to False to start over
    
    if RESUME_FROM_PREVIOUS:
        results_summary = previous_results
        print(f"\nResuming from previous run...")
    else:
        results_summary = []
        completed_experiments = []
        print(f"\nStarting fresh (ignoring previous progress)...")
else:
    results_summary = []
    completed_experiments = []
    print("No previous progress found. Starting fresh.")

Found previous progress: 10 completed experiments
baseline_s_640_e100
imgsz_512
imgsz_768
imgsz_1024
epochs_50
epochs_150
epochs_200
model_nano
model_medium
model_large

Resuming from previous run...


In [10]:
# Select which experiments to run
EXPERIMENTS_TO_RUN = None  # Run all (that haven't been completed)
# EXPERIMENTS_TO_RUN = [0]  # Run only baseline
# EXPERIMENTS_TO_RUN = [0, 1, 2, 3]  # Run baseline + image size experiments

# Whether to run SAHI evaluation after each training
RUN_SAHI_EVALUATION = True and SAHI_AVAILABLE

# Filter experiments
if EXPERIMENTS_TO_RUN is None:
    experiments_to_run = experiments
else:
    experiments_to_run = [experiments[i] for i in EXPERIMENTS_TO_RUN]

# Remove already completed experiments
experiments_to_run = [exp for exp in experiments_to_run 
                      if exp['name'] not in completed_experiments]

print(f"\nWill run {len(experiments_to_run)} experiment(s):")
for exp in experiments_to_run:
    print(f"  - {exp['name']}")

if not experiments_to_run:
    print("All experiments already completed!")

print(f"\nSAHI evaluation: {'Enabled' if RUN_SAHI_EVALUATION else 'Disabled'}")


Will run 10 experiment(s):
  - 796img_baseline_s_640_e100
  - 796img_imgsz_512
  - 796img_imgsz_768
  - 796img_imgsz_1024
  - 796img_epochs_50
  - 796img_epochs_150
  - 796img_epochs_200
  - 796img_model_nano
  - 796img_model_medium
  - 796img_model_large

SAHI evaluation: Enabled


## 6. Run Experiments

In [11]:
# Track total time
total_start_time = time.time()

print(f"Starting experiments at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"GPU Memory at start: {get_gpu_memory_usage()}")
print("=" * 70)

for i, exp in enumerate(experiments_to_run, 1):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT {i}/{len(experiments_to_run)}: {exp['name']}")
    print(f"Model: {exp['model']}")
    print(f"Changes from baseline: {exp['changes'] if exp['changes'] else 'None (baseline)'}")
    print(f"GPU Memory: {get_gpu_memory_usage()}")
    print(f"{'='*70}\n")
    
    # Clear memory before starting
    clear_memory()
    
    # Build config: start with base, apply changes
    config = BASE_CONFIG.copy()
    config.update(exp['changes'])
    config['name'] = exp['name']
    
    # Get the image size for this experiment (for SAHI slice size)
    imgsz = config['imgsz']
    
    # Track time for this experiment
    exp_start_time = time.time()
    
    # Initialize model variable
    model = None
    
    try:
        # =====================================================================
        # TRAINING
        # =====================================================================
        print(f"Training with imgsz={imgsz}, batch={config['batch']}...")
        model = YOLO(exp['model'])
        results = model.train(**config)
        
        train_time = time.time() - exp_start_time
        
        # Extract training metrics
        metrics = results.results_dict
        best_model_path = Path(results.save_dir) / 'weights' / 'best.pt'
        
        # Base result entry
        result_entry = {
            'experiment': exp['name'],
            'model': exp['model'],
            'imgsz': imgsz,
            'epochs': config['epochs'],
            'batch': config['batch'],
            'mAP50': metrics.get('metrics/mAP50(B)', None),
            'mAP50-95': metrics.get('metrics/mAP50-95(B)', None),
            'precision': metrics.get('metrics/precision(B)', None),
            'recall': metrics.get('metrics/recall(B)', None),
            'training_time_min': round(train_time / 60, 1),
            'best_model_path': str(best_model_path),
            'status': 'success'
        }
        
        print(f"Training completed in {train_time/60:.1f} min")
        print(f"   mAP50: {result_entry['mAP50']:.4f}")
        print(f"   Recall: {result_entry['recall']:.4f}")
        print(f"   Precision: {result_entry['precision']:.4f}")
        
        # Clean up training model
        del model
        del results
        model = None
        clear_memory()
        
        # =====================================================================
        # SAHI EVALUATION
        # =====================================================================
        if RUN_SAHI_EVALUATION:
            print(f"\n   Running SAHI evaluation (slice_size={imgsz})...")
            
            # Standard inference
            std_results = evaluate_standard(
                best_model_path,
                val_images_dir,
                imgsz=imgsz,
                conf_thresh=SAHI_CONFIDENCE_THRESHOLD
            )
            
            # SAHI inference
            sahi_results = evaluate_with_sahi(
                best_model_path,
                val_images_dir,
                slice_size=imgsz,
                overlap_ratio=SAHI_OVERLAP_RATIO,
                conf_thresh=SAHI_CONFIDENCE_THRESHOLD
            )
            
            # Add SAHI metrics to result
            if std_results:
                result_entry['std_detections'] = std_results['total_detections']
                result_entry['std_time_per_img'] = round(std_results['time_per_image'], 3)
            
            if sahi_results:
                result_entry['sahi_detections'] = sahi_results['total_detections']
                result_entry['sahi_time_per_img'] = round(sahi_results['time_per_image'], 3)
                result_entry['sahi_slice_size'] = imgsz
                
                if std_results and std_results['total_detections'] > 0:
                    detection_increase = sahi_results['total_detections'] - std_results['total_detections']
                    pct_increase = ((sahi_results['total_detections'] / std_results['total_detections']) - 1) * 100
                    result_entry['detection_increase'] = detection_increase
                    result_entry['detection_increase_pct'] = round(pct_increase, 1)
                    
                    print(f"   Standard detections: {std_results['total_detections']}")
                    print(f"   SAHI detections: {sahi_results['total_detections']} ({pct_increase:+.1f}%)")
        
    except Exception as e:
        exp_time = time.time() - exp_start_time
        print(f"FAILED: {exp['name']} after {exp_time/60:.1f} min")
        print(f"   Error: {str(e)}")
        
        result_entry = {
            'experiment': exp['name'],
            'model': exp['model'],
            'imgsz': imgsz,
            'epochs': config['epochs'],
            'batch': config['batch'],
            'mAP50': None,
            'mAP50-95': None,
            'precision': None,
            'recall': None,
            'training_time_min': round(exp_time / 60, 1),
            'status': f'failed: {str(e)[:100]}'
        }
    
    finally:
        # Always clean up
        if model is not None:
            del model
        clear_memory()
    
    # Save result and progress
    results_summary.append(result_entry)
    save_progress(results_summary, progress_file)
    
    print(f"\n   Completed {len(results_summary)}/{len(experiments)} total experiments")

# Total time
total_time = time.time() - total_start_time
print(f"\n{'='*70}")
print(f"ALL EXPERIMENTS COMPLETED")
print(f"Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"{'='*70}")

Starting experiments at 2026-02-25 10:38:26
GPU Memory at start: 0.0/191.5 GB

EXPERIMENT 1/10: 796img_baseline_s_640_e100
Model: yolo26s.pt
Changes from baseline: None (baseline)
GPU Memory: 0.0/191.5 GB

   Memory cleared.
Training with imgsz=640, batch=16...
New https://pypi.org/project/ultralytics/8.4.16 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA B200, 182642MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, fli

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_baseline_s_640_e1002/labels.jpg... 
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_baseline_s_640_e1002
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.06G      2.858      3.106   0.003959         27        640: 100% ━━━━━━━━━━━━ 40/40 2.3it/s 17.5s0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.2s0.3ss
     

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_imgsz_512/labels.jpg... 
Image sizes 512 train, 512 val
Using 4 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_imgsz_512
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      1.51G       3.05       3.28   0.004698         24        512: 100% ━━━━━━━━━━━━ 40/40 2.3it/s 17.3s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.1s0.3ss
                   all     

...

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_model_nano/labels.jpg... 
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_model_nano
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100       1.1G      3.311      6.458   0.005756         27        640: 100% ━━━━━━━━━━━━ 40/40 3.7it/s 10.8s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 4.6it/s 1.1s0.3ss
                   all   

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 124 weight(decay=0.0), 136 weight(decay=0.0005), 136 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_model_medium/labels.jpg... 
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_model_medium
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.14G      2.794      4.227    0.00362         12        640: 100% ━━━━━━━━━━━━ 80/80 5.5it/s 14.4s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.6it/s 1.8s0.1s
                   a

/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/aileenlavelle/.local/lib/python3.10/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 178 weight(decay=0.0), 190 weight(decay=0.0005), 190 bias(decay=0.0)
Plotting labels to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_model_large/labels.jpg... 
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_experiments_v2/796img_model_large
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100       1.6G      2.757      4.754   0.003579         11        640: 100% ━━━━━━━━━━━━ 159/159 13.6it/s 11.7s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 20/20 9.5it/s 2.1s<0.0s
                   

## 7. Results Summary

In [12]:
try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [13]:
# Display training results
print("\n" + "="*80)
print("TRAINING RESULTS SUMMARY")
print("="*80 + "\n")

df = pd.DataFrame(results_summary)

# Format for display
train_cols = ['experiment', 'model', 'imgsz', 'epochs', 'batch', 'mAP50', 'recall', 'precision', 'training_time_min', 'status']
train_cols = [c for c in train_cols if c in df.columns]
df_train = df[train_cols].copy()

# Round numeric columns
for col in ['mAP50', 'recall', 'precision']:
    if col in df_train.columns:
        df_train[col] = df_train[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) and x is not None else "N/A")

print(df_train.to_string(index=False))


TRAINING RESULTS SUMMARY

                experiment      model  imgsz  epochs  batch  mAP50 recall precision  training_time_min  status
       baseline_s_640_e100 yolo26s.pt    640     100     16 0.6609 0.6120    0.7291                2.9 success
                 imgsz_512 yolo26s.pt    512     100     16 0.5397 0.4667    0.7351                2.7 success
                 imgsz_768 yolo26s.pt    768     100      8 0.7227 0.6627    0.7391                4.1 success
                imgsz_1024 yolo26s.pt   1024     100      4 0.8208 0.7362    0.8583                7.0 success
                 epochs_50 yolo26s.pt    640      50     16 0.6559 0.6275    0.6972                1.4 success
                epochs_150 yolo26s.pt    640     150     16 0.6444 0.5451    0.8185                3.8 success
                epochs_200 yolo26s.pt    640     200     16 0.6282 0.5613    0.7690                4.7 success
                model_nano yolo26n.pt    640     100     16 0.5469 0.4941    0.7044  

In [14]:
# Display SAHI comparison results
if RUN_SAHI_EVALUATION and any('sahi_detections' in r for r in results_summary):
    print("\n" + "="*80)
    print("SAHI vs STANDARD INFERENCE COMPARISON")
    print("="*80 + "\n")
    
    if PANDAS_AVAILABLE:
        sahi_cols = ['experiment', 'imgsz', 'sahi_slice_size', 'std_detections', 'sahi_detections', 
                     'detection_increase', 'detection_increase_pct']
        sahi_cols = [c for c in sahi_cols if c in df.columns]
        df_sahi = df[sahi_cols].copy()
        print(df_sahi.to_string(index=False))
    else:
        print(f"{'Experiment':<25} {'Std Det':<10} {'SAHI Det':<10} {'Increase':<10}")
        print("-" * 60)
        for r in results_summary:
            if 'sahi_detections' in r:
                inc = r.get('detection_increase_pct', 'N/A')
                print(f"{r['experiment']:<25} {r.get('std_detections', 'N/A'):<10} {r['sahi_detections']:<10} {inc}%")


SAHI vs STANDARD INFERENCE COMPARISON

                experiment  imgsz  sahi_slice_size  std_detections  sahi_detections  detection_increase  detection_increase_pct
       baseline_s_640_e100    640              640             230              445                 215                    93.5
                 imgsz_512    512              512             184              388                 204                   110.9
                 imgsz_768    768              768             247              349                 102                    41.3
                imgsz_1024   1024             1024             268              318                  50                    18.7
                 epochs_50    640              640             238              572                 334                   140.3
                epochs_150    640              640             201              408                 207                   103.0
                epochs_200    640              640             1

In [15]:
# Find best performing experiments
successful_results = [r for r in results_summary if r.get('status') == 'success' and r.get('mAP50')]

if successful_results:
    print("\n" + "="*60)
    print("BEST PERFORMERS")
    print("="*60)
    
    # Best by mAP50
    best_map = max(successful_results, key=lambda x: x['mAP50'])
    print(f"Best mAP50: {best_map['experiment']}")
    print(f"   mAP50: {best_map['mAP50']:.4f}")
    print(f"   Config: {best_map['model']}, imgsz={best_map['imgsz']}, epochs={best_map['epochs']}")
    
    # Best by Recall
    best_recall = max(successful_results, key=lambda x: x['recall'])
    print(f"Best Recall: {best_recall['experiment']}")
    print(f"   Recall: {best_recall['recall']:.4f}")
    print(f"   Config: {best_recall['model']}, imgsz={best_recall['imgsz']}, epochs={best_recall['epochs']}")
    
    # Best by Precision
    best_prec = max(successful_results, key=lambda x: x['precision'])
    print(f"Best Precision: {best_prec['experiment']}")
    print(f"   Precision: {best_prec['precision']:.4f}")
    
    # Best SAHI improvement
    sahi_results = [r for r in successful_results if r.get('detection_increase')]
    if sahi_results:
        best_sahi = max(sahi_results, key=lambda x: x['detection_increase'])
        print(f"Best SAHI Improvement: {best_sahi['experiment']}")
        print(f"   Extra detections: +{best_sahi['detection_increase']} ({best_sahi['detection_increase_pct']}%)")
else:
    print("No successful experiments to analyze.")


BEST PERFORMERS
Best mAP50: 796img_imgsz_1024
   mAP50: 0.8574
   Config: yolo26s.pt, imgsz=1024, epochs=100
Best Recall: 796img_imgsz_1024
   Recall: 0.8040
   Config: yolo26s.pt, imgsz=1024, epochs=100
Best Precision: imgsz_1024
   Precision: 0.8583
Best SAHI Improvement: 796img_model_nano
   Extra detections: +762 (153.6%)


In [16]:
# Save final results to CSV
output_dir = Path(BASE_CONFIG['project'])
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save as JSON (always works)
json_path = output_dir / f'experiment_results_{timestamp}.json'
save_progress(results_summary, str(json_path))

# Save as CSV if pandas available
if PANDAS_AVAILABLE:
    csv_path = output_dir / f'experiment_results_{timestamp}.csv'
    df.to_csv(csv_path, index=False)
    print(f"CSV saved to: {csv_path}")

print(f"JSON saved to: {json_path}")

   Progress saved to jupiter_inlet_experiments_v2/experiment_results_20260225_123754.json
CSV saved to: jupiter_inlet_experiments_v2/experiment_results_20260225_123754.csv
JSON saved to: jupiter_inlet_experiments_v2/experiment_results_20260225_123754.json


In [1]:
"""
SAHI Metrics & Visualization
=============================
Drop these functions into your existing notebook.

Functions:
  - evaluate_with_sahi_metrics()  →  computes Precision, Recall, mAP50 vs ground truth
  - visualize_sahi_predictions()  →  dark-blue GT boxes + orange SAHI boxes side-by-side
  - run_sahi_viz_sample()         →  convenience wrapper to visualise N random val images
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import torch
import time
import gc


# ──────────────────────────────────────────────────────────────────────────────
# 1.  METRICS HELPERS
# ──────────────────────────────────────────────────────────────────────────────

def _xywhn_to_xyxy(boxes_norm, img_w, img_h):
    """Convert YOLO normalised cx,cy,w,h → x1,y1,x2,y2 in pixels."""
    if len(boxes_norm) == 0:
        return np.empty((0, 4), dtype=np.float32)
    b = np.asarray(boxes_norm, dtype=np.float32)
    cx, cy, bw, bh = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    x1 = (cx - bw / 2) * img_w
    y1 = (cy - bh / 2) * img_h
    x2 = (cx + bw / 2) * img_w
    y2 = (cy + bh / 2) * img_h
    return np.stack([x1, y1, x2, y2], axis=1)


def _iou_matrix(pred_boxes, gt_boxes):
    """Compute IoU between every pair (N_pred x N_gt)."""
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return np.zeros((len(pred_boxes), len(gt_boxes)), dtype=np.float32)
    px1, py1, px2, py2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    gx1, gy1, gx2, gy2 = gt_boxes[:, 0],  gt_boxes[:, 1],  gt_boxes[:, 2],  gt_boxes[:, 3]
    inter_x1 = np.maximum(px1[:, None], gx1[None, :])
    inter_y1 = np.maximum(py1[:, None], gy1[None, :])
    inter_x2 = np.minimum(px2[:, None], gx2[None, :])
    inter_y2 = np.minimum(py2[:, None], gy2[None, :])
    inter_w  = np.maximum(0, inter_x2 - inter_x1)
    inter_h  = np.maximum(0, inter_y2 - inter_y1)
    inter    = inter_w * inter_h
    area_p   = (px2 - px1) * (py2 - py1)
    area_g   = (gx2 - gx1) * (gy2 - gy1)
    union    = area_p[:, None] + area_g[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


def _compute_ap(recalls, precisions):
    """Compute AP using the 11-point interpolation (VOC style)."""
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        p = precisions[recalls >= t]
        ap += (p.max() if len(p) > 0 else 0.0)
    return ap / 11.0


def _pr_curve_for_image(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5):
    """Return sorted (score, tp, fp) rows for one image."""
    rows = []
    matched_gt = set()
    if len(pred_boxes) == 0:
        return rows, len(gt_boxes)
    iou = _iou_matrix(pred_boxes, gt_boxes)   # (N_pred, N_gt)
    order = np.argsort(-pred_scores)
    for idx in order:
        score = pred_scores[idx]
        if len(gt_boxes) == 0:
            rows.append((score, 0, 1))
            continue
        best_gt = int(np.argmax(iou[idx]))
        if iou[idx, best_gt] >= iou_thresh and best_gt not in matched_gt:
            matched_gt.add(best_gt)
            rows.append((score, 1, 0))
        else:
            rows.append((score, 0, 1))
    return rows, len(gt_boxes)


def _load_gt_labels(label_path, img_w, img_h):
    """Load YOLO txt label file → pixel xyxy boxes."""
    if not Path(label_path).exists():
        return np.empty((0, 4), dtype=np.float32)
    rows = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                rows.append([float(p) for p in parts[1:5]])  # skip class id
    return _xywhn_to_xyxy(rows, img_w, img_h)


def _get_image_size(img_path):
    """Return (width, height) of an image without loading full pixels."""
    from PIL import Image
    with Image.open(img_path) as im:
        return im.size   # (w, h)


# ──────────────────────────────────────────────────────────────────────────────
# 2.  MAIN METRICS FUNCTION  (drop-in replacement / addition to your notebook)
# ──────────────────────────────────────────────────────────────────────────────

def evaluate_with_sahi_metrics(
    model_path,
    val_images_dir,
    val_labels_dir,          # <-- NEW: path to your YOLO label .txt files
    slice_size,
    overlap_ratio=0.2,
    conf_thresh=0.25,
    iou_thresh=0.5,
):
    """
    Run SAHI sliced inference on the val set and compute:
      • Precision, Recall at conf_thresh
      • mAP50 (Pascal-VOC 11-point interpolation)

    Parameters
    ----------
    model_path     : path to best.pt
    val_images_dir : directory containing validation images
    val_labels_dir : directory containing matching YOLO .txt label files
    slice_size     : SAHI slice height/width in pixels
    overlap_ratio  : SAHI overlap (default 0.2)
    conf_thresh    : confidence threshold for detections
    iou_thresh     : IoU threshold for TP/FP matching (default 0.5 = mAP50)

    Returns
    -------
    dict with keys: precision, recall, mAP50, total_detections,
                    num_images, inference_time, time_per_image
    """
    try:
        from sahi import AutoDetectionModel
        from sahi.predict import get_sliced_prediction
        from PIL import Image
    except ImportError:
        print("   sahi / Pillow not installed.")
        return None

    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=str(model_path),
        confidence_threshold=conf_thresh,
        device='cuda:0' if torch.cuda.is_available() else 'cpu',
    )

    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    image_files = [
        f for f in Path(val_images_dir).iterdir()
        if f.suffix.lower() in image_extensions
    ]

    all_rows   = []   # (score, tp, fp)
    total_gt   = 0
    total_dets = 0
    start_time = time.time()

    for img_path in image_files:
        # ── ground truth ──────────────────────────────────────────────────────
        label_path = Path(val_labels_dir) / (img_path.stem + '.txt')
        img_w, img_h = _get_image_size(img_path)
        gt_boxes = _load_gt_labels(label_path, img_w, img_h)   # (N_gt, 4)

        # ── SAHI prediction ───────────────────────────────────────────────────
        result = get_sliced_prediction(
            str(img_path),
            detection_model,
            slice_height=slice_size,
            slice_width=slice_size,
            overlap_height_ratio=overlap_ratio,
            overlap_width_ratio=overlap_ratio,
            verbose=0,
        )

        preds      = result.object_prediction_list
        total_dets += len(preds)

        if len(preds) == 0:
            total_gt += len(gt_boxes)
            continue

        pred_boxes  = np.array(
            [[p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy] for p in preds],
            dtype=np.float32,
        )
        pred_scores = np.array([p.score.value for p in preds], dtype=np.float32)

        rows, n_gt = _pr_curve_for_image(pred_boxes, pred_scores, gt_boxes, iou_thresh)
        all_rows.extend(rows)
        total_gt += n_gt

    elapsed = time.time() - start_time

    # ── aggregate PR curve & AP ───────────────────────────────────────────────
    if len(all_rows) == 0 or total_gt == 0:
        del detection_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            'precision': 0.0, 'recall': 0.0, 'mAP50': 0.0,
            'total_detections': total_dets, 'num_images': len(image_files),
            'inference_time': elapsed, 'time_per_image': elapsed / max(1, len(image_files)),
        }

    all_rows   = sorted(all_rows, key=lambda r: -r[0])   # sort by score desc
    scores     = np.array([r[0] for r in all_rows])
    tps        = np.cumsum([r[1] for r in all_rows], dtype=np.float32)
    fps        = np.cumsum([r[2] for r in all_rows], dtype=np.float32)

    recalls    = tps / (total_gt + 1e-9)
    precisions = tps / (tps + fps + 1e-9)

    # At-threshold precision/recall (use the last row where score >= conf_thresh)
    mask = scores >= conf_thresh
    if mask.any():
        final_prec = float(precisions[mask][-1])
        final_rec  = float(recalls[mask][-1])
    else:
        final_prec, final_rec = 0.0, 0.0

    mAP50 = float(_compute_ap(recalls, precisions))

    del detection_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'precision':         round(final_prec, 4),
        'recall':            round(final_rec,  4),
        'mAP50':             round(mAP50,       4),
        'total_detections':  total_dets,
        'num_images':        len(image_files),
        'inference_time':    round(elapsed, 2),
        'time_per_image':    round(elapsed / max(1, len(image_files)), 3),
    }


# ──────────────────────────────────────────────────────────────────────────────
# 3.  VISUALISATION FUNCTION
# ──────────────────────────────────────────────────────────────────────────────

GT_COLOR   = '#003f8a'   # dark navy blue  ── ground truth
PRED_COLOR = '#ff6b00'   # vivid orange    ── SAHI predictions
GT_LABEL   = 'Ground Truth'
PRED_LABEL = 'SAHI Prediction'


def visualize_sahi_predictions(
    model_path,
    val_images_dir,
    val_labels_dir,
    slice_size,
    overlap_ratio=0.2,
    conf_thresh=0.25,
    n_images=4,
    seed=42,
    save_path=None,          # optional: path to save the figure (e.g. 'sahi_viz.png')
    figsize_per_image=(6, 6),
):
    """
    Plot N random val images with:
      ■ Dark-blue  boxes = ground truth annotations
      ■ Orange     boxes = SAHI model predictions

    Parameters
    ----------
    model_path     : path to best.pt
    val_images_dir : directory of val images
    val_labels_dir : directory of matching YOLO .txt labels
    slice_size     : SAHI slice size in pixels
    n_images       : how many images to visualise (default 4)
    save_path      : if given, also saves the figure to this path
    """
    try:
        from sahi import AutoDetectionModel
        from sahi.predict import get_sliced_prediction
        from PIL import Image
    except ImportError:
        print("sahi / Pillow not installed.")
        return

    # ── load model ────────────────────────────────────────────────────────────
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=str(model_path),
        confidence_threshold=conf_thresh,
        device='cuda:0' if torch.cuda.is_available() else 'cpu',
    )

    # ── pick images ───────────────────────────────────────────────────────────
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    image_files = [
        f for f in Path(val_images_dir).iterdir()
        if f.suffix.lower() in image_extensions
    ]
    rng = np.random.default_rng(seed)
    chosen = rng.choice(image_files, size=min(n_images, len(image_files)), replace=False)

    # ── build figure ──────────────────────────────────────────────────────────
    ncols = min(2, len(chosen))
    nrows = int(np.ceil(len(chosen) / ncols))
    fig_w = figsize_per_image[0] * ncols
    fig_h = figsize_per_image[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(-1)   # flatten for easy indexing

    for i, img_path in enumerate(chosen):
        ax = axes[i]

        # load image
        img = np.array(Image.open(img_path).convert('RGB'))
        img_h, img_w = img.shape[:2]

        # load GT
        label_path = Path(val_labels_dir) / (img_path.stem + '.txt')
        gt_boxes   = _load_gt_labels(label_path, img_w, img_h)

        # SAHI prediction
        result = get_sliced_prediction(
            str(img_path),
            detection_model,
            slice_height=slice_size,
            slice_width=slice_size,
            overlap_height_ratio=overlap_ratio,
            overlap_width_ratio=overlap_ratio,
            verbose=0,
        )
        preds = result.object_prediction_list

        # ── draw ──────────────────────────────────────────────────────────────
        ax.imshow(img)
        ax.set_title(
            f'{img_path.name}\n'
            f'GT: {len(gt_boxes)}  |  Pred: {len(preds)}',
            fontsize=9, pad=4,
        )
        ax.axis('off')

        # ground truth – dark blue, solid line
        for box in gt_boxes:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=GT_COLOR, facecolor='none',
                label=GT_LABEL,
            )
            ax.add_patch(rect)

        # predictions – orange, dashed line
        for pred in preds:
            x1 = pred.bbox.minx
            y1 = pred.bbox.miny
            x2 = pred.bbox.maxx
            y2 = pred.bbox.maxy
            score = pred.score.value
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=PRED_COLOR, facecolor='none',
                linestyle='--', label=PRED_LABEL,
            )
            ax.add_patch(rect)
            ax.text(
                x1, max(y1 - 4, 0),
                f'{score:.2f}',
                color=PRED_COLOR, fontsize=7,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.5, ec='none'),
            )

    # hide unused axes
    for j in range(len(chosen), len(axes)):
        axes[j].set_visible(False)

    # ── shared legend ─────────────────────────────────────────────────────────
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color=GT_COLOR,   linewidth=2,              label=f'{GT_LABEL} (solid)'),
        Line2D([0], [0], color=PRED_COLOR, linewidth=2, linestyle='--', label=f'{PRED_LABEL} (dashed)'),
    ]
    fig.legend(
        handles=legend_elements,
        loc='lower center',
        ncol=2,
        fontsize=10,
        frameon=True,
        bbox_to_anchor=(0.5, -0.01),
    )

    fig.suptitle(
        f'SAHI Predictions vs Ground Truth\n'
        f'slice={slice_size}px  overlap={overlap_ratio}  conf≥{conf_thresh}',
        fontsize=12, fontweight='bold', y=1.01,
    )
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"   Figure saved to {save_path}")

    plt.show()

    # clean up
    del detection_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ──────────────────────────────────────────────────────────────────────────────
# 4.  CONVENIENCE RUNNER — paste this block into a new notebook cell
# ──────────────────────────────────────────────────────────────────────────────
#
# USAGE EXAMPLE (edit these four paths/values to match your setup):
#
# BEST_MODEL   = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/" \
#                "jupiter_inlet_experiments_v2/796img_baseline_s_640_e1002/weights/best.pt"
#
# VAL_IMAGES   = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/" \
#                "Jupiter_Inlet/jupiter_inlet_yolo/images/val"
#
# VAL_LABELS   = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/" \
#                "Jupiter_Inlet/jupiter_inlet_yolo/labels/val"
#
# SLICE_SIZE   = 640   # match the imgsz used during training
#
# # ── compute SAHI metrics ──────────────────────────────────────────────────
# sahi_metrics = evaluate_with_sahi_metrics(
#     model_path     = BEST_MODEL,
#     val_images_dir = VAL_IMAGES,
#     val_labels_dir = VAL_LABELS,
#     slice_size     = SLICE_SIZE,
#     overlap_ratio  = 0.2,
#     conf_thresh    = 0.25,
#     iou_thresh     = 0.5,
# )
# print("\nSAHI Metrics:")
# for k, v in sahi_metrics.items():
#     print(f"  {k:30s}: {v}")
#
# # ── visualise random val images ───────────────────────────────────────────
# visualize_sahi_predictions(
#     model_path     = BEST_MODEL,
#     val_images_dir = VAL_IMAGES,
#     val_labels_dir = VAL_LABELS,
#     slice_size     = SLICE_SIZE,
#     n_images       = 6,          # how many images to show
#     seed           = 42,
#     save_path      = "sahi_viz.png",   # set to None to skip saving
# )

## 8. Visualize Results

In [17]:
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("Matplotlib not available, skipping plots")

In [18]:
if MATPLOTLIB_AVAILABLE and successful_results:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    exp_names = [r['experiment'] for r in successful_results]
    mAP50s = [r['mAP50'] for r in successful_results]
    recalls = [r['recall'] for r in successful_results]
    precisions = [r['precision'] for r in successful_results]
    
    # mAP50 comparison
    axes[0].barh(exp_names, mAP50s, color='steelblue')
    axes[0].set_xlabel('mAP50')
    axes[0].set_title('mAP50 by Experiment')
    axes[0].axvline(x=sum(mAP50s)/len(mAP50s), color='red', linestyle='--', label='Mean')
    
    # Recall comparison
    axes[1].barh(exp_names, recalls, color='forestgreen')
    axes[1].set_xlabel('Recall')
    axes[1].set_title('Recall by Experiment')
    axes[1].axvline(x=sum(recalls)/len(recalls), color='red', linestyle='--', label='Mean')
    
    # Precision comparison
    axes[2].barh(exp_names, precisions, color='darkorange')
    axes[2].set_xlabel('Precision')
    axes[2].set_title('Precision by Experiment')
    axes[2].axvline(x=sum(precisions)/len(precisions), color='red', linestyle='--', label='Mean')
    
    plt.tight_layout()
    
    # Save figure
    fig_path = output_dir / 'experiment_comparison.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"Figure saved to: {fig_path}")
    
    plt.show()
else:
    print("Skipping plots (matplotlib not available or no successful experiments)")

Figure saved to: jupiter_inlet_experiments_v2/experiment_comparison.png


<Figure size 1500x500 with 3 Axes>

In [19]:
# SAHI comparison plot
if MATPLOTLIB_AVAILABLE and RUN_SAHI_EVALUATION:
    sahi_results_plot = [r for r in successful_results if r.get('sahi_detections') and r.get('std_detections')]
    
    if sahi_results_plot:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        exp_names = [r['experiment'] for r in sahi_results_plot]
        std_dets = [r['std_detections'] for r in sahi_results_plot]
        sahi_dets = [r['sahi_detections'] for r in sahi_results_plot]
        increases = [r.get('detection_increase', 0) for r in sahi_results_plot]
        
        # Detection counts: Standard vs SAHI
        x = range(len(exp_names))
        width = 0.35
        
        axes[0].bar([i - width/2 for i in x], std_dets, width, label='Standard', color='steelblue')
        axes[0].bar([i + width/2 for i in x], sahi_dets, width, label='SAHI', color='coral')
        axes[0].set_xlabel('Experiment')
        axes[0].set_ylabel('Total Detections')
        axes[0].set_title('Standard vs SAHI Detections')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(exp_names, rotation=45, ha='right')
        axes[0].legend()
        
        # Detection increase
        colors = ['forestgreen' if inc >= 0 else 'red' for inc in increases]
        axes[1].bar(exp_names, increases, color=colors)
        axes[1].set_xlabel('Experiment')
        axes[1].set_ylabel('Additional Detections (SAHI - Standard)')
        axes[1].set_title('SAHI Detection Improvement')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
        
        plt.tight_layout()
        
        # Save figure
        fig_path = output_dir / 'sahi_comparison.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to: {fig_path}")
        
        plt.show()

Figure saved to: jupiter_inlet_experiments_v2/sahi_comparison.png


<Figure size 1400x500 with 2 Axes>